# AHFoZ Federated Fraud Detection System
## Association of Healthcare Funders of Zimbabwe
### Data Analytics & Fraud Detection Pipeline

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Configuration & Parameters

In [ ]:
# Fraud detection thresholds
OUTLIER_THRESHOLD_SD = 3.0
QUANTITY_THRESHOLD = 30
TEMPORAL_WINDOW_HOURS = 24
MIN_CLAIMS_FOR_ANALYSIS = 10

# Column mapping (adjust to your CSV structure)
REQUIRED_COLUMNS = [
    'Claim_ID',
    'Provider_ID', 
    'Provider_Name',
    'Member_ID',
    'Member_Name',
    'Service_Date',
    'Amount',
    'Quantity',
    'Drug_Code'
]

# Node information
NODE_NAME = 'Fund_A'
NODE_ID = 'NODE_' + str(np.random.randint(100000, 999999))

## 3. Data Loading & Validation

In [ ]:
def load_and_validate_data(filepath):
    """
    Load CSV data and perform initial validation
    """
    print(f"Loading data from: {filepath}")
    
    # Load the CSV
    df = pd.read_csv(filepath)
    
    print(f"Loaded {len(df):,} records")
    print(f"Columns: {list(df.columns)}")
    
    # Check for required columns
    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    
    if missing_cols:
        print(f"Missing columns: {missing_cols}")
        print("Available columns:", list(df.columns))
    
    # Convert date column to datetime
    if 'Service_Date' in df.columns:
        df['Service_Date'] = pd.to_datetime(df['Service_Date'], errors='coerce')
    
    # Convert numeric columns
    if 'Amount' in df.columns:
        df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
    
    if 'Quantity' in df.columns:
        df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
    
    # Remove rows with missing critical data
    df_clean = df.dropna(subset=['Claim_ID', 'Amount'], how='any')
    
    removed = len(df) - len(df_clean)
    if removed > 0:
        print(f"Removed {removed} rows with missing critical data")
    
    print(f"Clean dataset: {len(df_clean):,} records")
    
    return df_clean

In [ ]:
# Load your data - REPLACE WITH YOUR ACTUAL FILE PATH
data_file = 'claims_data.csv'
df = load_and_validate_data(data_file)

## 4. Exploratory Data Analysis

In [ ]:
def generate_summary_statistics(df):
    """
    Generate comprehensive summary statistics
    """
    stats = {}
    
    # Basic counts
    stats['total_records'] = len(df)
    stats['total_claims'] = df['Claim_ID'].nunique()
    stats['total_providers'] = df['Provider_ID'].nunique()
    stats['total_members'] = df['Member_ID'].nunique()
    
    # Financial metrics
    stats['total_amount'] = df['Amount'].sum()
    stats['avg_claim_amount'] = df['Amount'].mean()
    stats['median_claim_amount'] = df['Amount'].median()
    stats['std_claim_amount'] = df['Amount'].std()
    
    # Quantity metrics
    if 'Quantity' in df.columns:
        stats['avg_quantity'] = df['Quantity'].mean()
        stats['max_quantity'] = df['Quantity'].max()
    
    # Temporal metrics
    if 'Service_Date' in df.columns:
        stats['date_range_start'] = df['Service_Date'].min()
        stats['date_range_end'] = df['Service_Date'].max()
        stats['date_span_days'] = (stats['date_range_end'] - stats['date_range_start']).days
    
    return stats

In [ ]:
# Generate and display summary statistics
summary_stats = generate_summary_statistics(df)

print("="*60)
print("DATASET SUMMARY STATISTICS")
print("="*60)

for key, value in summary_stats.items():
    if isinstance(value, (int, np.integer)):
        print(f"{key}: {value:,}")
    elif isinstance(value, (float, np.floating)):
        print(f"{key}: ${value:,.2f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Display dataset info
df.info()

In [ ]:
# Display first 10 records
df.head(10)

In [ ]:
# Statistical summary
df.describe()

## 5. Fraud Detection Functions

### 5.1 Duplicate Claims Detection

In [ ]:
def detect_duplicate_claims(df):
    """
    Detect exact duplicate claims
    """
    print("DUPLICATE CLAIMS DETECTION")
    print("="*60)
    
    dup_cols = ['Provider_ID', 'Member_ID', 'Service_Date', 'Amount']
    duplicates = df[df.duplicated(subset=dup_cols, keep=False)]
    
    if len(duplicates) == 0:
        print("No duplicate claims found")
        return pd.DataFrame()
    
    duplicates = duplicates.sort_values(by=dup_cols)
    dup_groups = duplicates.groupby(dup_cols).size().reset_index(name='Count')
    
    print(f"Found {len(dup_groups)} duplicate claim groups")
    print(f"Total duplicate records: {len(duplicates)}")
    print(f"Potential duplicated amount: ${duplicates['Amount'].sum():,.2f}")
    
    alerts = []
    for idx, row in duplicates.iterrows():
        alert = {
            'Claim_ID': row['Claim_ID'],
            'Provider': row['Provider_Name'],
            'Provider_ID': row['Provider_ID'],
            'Member': row['Member_Name'],
            'Member_ID': row['Member_ID'],
            'Amount': row['Amount'],
            'Service_Date': row['Service_Date'],
            'Check': 'Duplicate Claim',
            'Severity': 'HIGH',
            'Finding': 'Exact duplicate: same provider, member, date, and amount'
        }
        alerts.append(alert)
    
    return pd.DataFrame(alerts)

### 5.2 Amount Outlier Detection

In [ ]:
def detect_amount_outliers(df, threshold_sd=3.0):
    """
    Detect claims with amounts beyond N standard deviations
    """
    print(f"AMOUNT OUTLIERS DETECTION ({threshold_sd} sigma)")
    print("="*60)
    
    mean_amount = df['Amount'].mean()
    std_amount = df['Amount'].std()
    upper_threshold = mean_amount + (threshold_sd * std_amount)
    lower_threshold = mean_amount - (threshold_sd * std_amount)
    
    print(f"Mean amount: ${mean_amount:,.2f}")
    print(f"Std deviation: ${std_amount:,.2f}")
    print(f"Upper threshold: ${upper_threshold:,.2f}")
    
    outliers = df[(df['Amount'] > upper_threshold) | (df['Amount'] < lower_threshold)]
    
    if len(outliers) == 0:
        print(f"No amount outliers found beyond {threshold_sd} sigma")
        return pd.DataFrame()
    
    print(f"Found {len(outliers)} outlier claims")
    print(f"Total outlier amount: ${outliers['Amount'].sum():,.2f}")
    
    alerts = []
    for idx, row in outliers.iterrows():
        deviation = abs(row['Amount'] - mean_amount) / std_amount
        alert = {
            'Claim_ID': row['Claim_ID'],
            'Provider': row['Provider_Name'],
            'Provider_ID': row['Provider_ID'],
            'Member': row['Member_Name'],
            'Member_ID': row['Member_ID'],
            'Amount': row['Amount'],
            'Service_Date': row['Service_Date'],
            'Check': 'Amount Outlier',
            'Severity': 'MEDIUM',
            'Finding': f'Amount ${row["Amount"]:,.2f} is {deviation:.1f} sigma from mean'
        }
        alerts.append(alert)
    
    return pd.DataFrame(alerts)

### 5.3 Quantity Threshold Detection

In [ ]:
def detect_quantity_threshold(df, threshold=30):
    """
    Detect claims with unusually high quantities
    """
    print(f"QUANTITY THRESHOLD DETECTION (>{threshold})")
    print("="*60)
    
    if 'Quantity' not in df.columns:
        print("Quantity column not found")
        return pd.DataFrame()
    
    high_qty = df[df['Quantity'] > threshold]
    
    if len(high_qty) == 0:
        print(f"No claims exceed quantity threshold of {threshold}")
        return pd.DataFrame()
    
    print(f"Found {len(high_qty)} claims with quantity > {threshold}")
    print(f"Max quantity: {high_qty['Quantity'].max():.0f}")
    
    alerts = []
    for idx, row in high_qty.iterrows():
        alert = {
            'Claim_ID': row['Claim_ID'],
            'Provider': row['Provider_Name'],
            'Provider_ID': row['Provider_ID'],
            'Member': row['Member_Name'],
            'Member_ID': row['Member_ID'],
            'Amount': row['Amount'],
            'Service_Date': row['Service_Date'],
            'Quantity': row['Quantity'],
            'Check': 'High Quantity',
            'Severity': 'MEDIUM',
            'Finding': f'Quantity {row["Quantity"]:.0f} exceeds threshold {threshold}'
        }
        alerts.append(alert)
    
    return pd.DataFrame(alerts)

### 5.4 Temporal Pattern Detection

In [ ]:
def detect_temporal_patterns(df, window_hours=24):
    """
    Detect multiple claims from same member within short time window
    """
    print(f"TEMPORAL PATTERN DETECTION (within {window_hours}h)")
    print("="*60)
    
    if 'Service_Date' not in df.columns:
        print("Service_Date column not found")
        return pd.DataFrame()
    
    df_sorted = df.sort_values(['Member_ID', 'Service_Date'])
    alerts = []
    
    for member_id, group in df_sorted.groupby('Member_ID'):
        if len(group) < 2:
            continue
        
        group = group.reset_index(drop=True)
        
        for i in range(len(group) - 1):
            time_diff = (group.loc[i+1, 'Service_Date'] - group.loc[i, 'Service_Date'])
            hours_diff = time_diff.total_seconds() / 3600
            
            if 0 < hours_diff <= window_hours:
                alert = {
                    'Claim_ID': group.loc[i+1, 'Claim_ID'],
                    'Provider': group.loc[i+1, 'Provider_Name'],
                    'Provider_ID': group.loc[i+1, 'Provider_ID'],
                    'Member': group.loc[i+1, 'Member_Name'],
                    'Member_ID': member_id,
                    'Amount': group.loc[i+1, 'Amount'],
                    'Service_Date': group.loc[i+1, 'Service_Date'],
                    'Check': 'Temporal Pattern',
                    'Severity': 'LOW',
                    'Finding': f'Claim submitted {hours_diff:.1f}h after previous claim'
                }
                alerts.append(alert)
    
    if len(alerts) == 0:
        print(f"No temporal patterns detected within {window_hours}h window")
        return pd.DataFrame()
    
    print(f"Found {len(alerts)} temporal pattern flags")
    return pd.DataFrame(alerts)

### 5.5 Ghost Provider Detection

In [ ]:
def detect_ghost_providers(df, min_claims=3):
    """
    Detect providers with suspiciously low claim counts
    """
    print(f"GHOST PROVIDER DETECTION (min {min_claims} claims)")
    print("="*60)
    
    provider_stats = df.groupby(['Provider_ID', 'Provider_Name']).agg({
        'Claim_ID': 'count',
        'Amount': ['sum', 'mean'],
        'Member_ID': 'nunique'
    }).reset_index()
    
    provider_stats.columns = ['Provider_ID', 'Provider_Name', 'Claim_Count', 
                               'Total_Amount', 'Avg_Amount', 'Unique_Members']
    
    mean_claim_amount = df['Amount'].mean()
    
    suspicious = provider_stats[
        (provider_stats['Claim_Count'] < min_claims) & 
        (provider_stats['Avg_Amount'] > mean_claim_amount * 2)
    ]
    
    if len(suspicious) == 0:
        print("No suspicious providers detected")
        return pd.DataFrame()
    
    print(f"Found {len(suspicious)} potentially ghost providers")
    
    alerts = []
    for _, prov in suspicious.iterrows():
        prov_claims = df[df['Provider_ID'] == prov['Provider_ID']]
        
        for idx, row in prov_claims.iterrows():
            alert = {
                'Claim_ID': row['Claim_ID'],
                'Provider': row['Provider_Name'],
                'Provider_ID': row['Provider_ID'],
                'Member': row['Member_Name'],
                'Member_ID': row['Member_ID'],
                'Amount': row['Amount'],
                'Service_Date': row['Service_Date'],
                'Check': 'Ghost Provider',
                'Severity': 'HIGH',
                'Finding': f'Provider has only {prov["Claim_Count"]} claims but high avg amount'
            }
            alerts.append(alert)
    
    return pd.DataFrame(alerts)

## 6. Run All Fraud Detection Checks

In [ ]:
def run_all_fraud_checks(df):
    """
    Execute all fraud detection algorithms
    """
    print("FRAUD DETECTION ANALYSIS")
    print("="*60)
    
    all_alerts = []
    
    # Run all checks
    dup_alerts = detect_duplicate_claims(df)
    if not dup_alerts.empty:
        all_alerts.append(dup_alerts)
    
    outlier_alerts = detect_amount_outliers(df, OUTLIER_THRESHOLD_SD)
    if not outlier_alerts.empty:
        all_alerts.append(outlier_alerts)
    
    qty_alerts = detect_quantity_threshold(df, QUANTITY_THRESHOLD)
    if not qty_alerts.empty:
        all_alerts.append(qty_alerts)
    
    temporal_alerts = detect_temporal_patterns(df, TEMPORAL_WINDOW_HOURS)
    if not temporal_alerts.empty:
        all_alerts.append(temporal_alerts)
    
    ghost_alerts = detect_ghost_providers(df, min_claims=3)
    if not ghost_alerts.empty:
        all_alerts.append(ghost_alerts)
    
    # Combine all alerts
    if all_alerts:
        combined_alerts = pd.concat(all_alerts, ignore_index=True)
    else:
        combined_alerts = pd.DataFrame()
    
    print("FRAUD DETECTION SUMMARY")
    print("="*60)
    print(f"Total alerts: {len(combined_alerts)}")
    
    if not combined_alerts.empty:
        print("Alerts by severity:")
        print(combined_alerts['Severity'].value_counts())
        print("Alerts by check type:")
        print(combined_alerts['Check'].value_counts())
        print(f"Total flagged amount: ${combined_alerts['Amount'].sum():,.2f}")
    
    return combined_alerts

In [ ]:
# Execute all fraud detection checks
fraud_alerts = run_all_fraud_checks(df)

## 7. Display Results

In [ ]:
# Display top 20 alerts
if not fraud_alerts.empty:
    print("Top 20 Fraud Alerts:")
    fraud_alerts.head(20)
else:
    print("No fraud alerts - dataset appears clean")

## 8. Visualizations

In [ ]:
# Alerts by severity and check type
if not fraud_alerts.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    fraud_alerts['Severity'].value_counts().plot(
        kind='bar',
        ax=axes[0],
        color=['red', 'orange', 'yellow']
    )
    axes[0].set_title('Alerts by Severity')
    axes[0].set_xlabel('Severity')
    axes[0].set_ylabel('Count')
    
    fraud_alerts['Check'].value_counts().plot(
        kind='barh',
        ax=axes[1],
        color='steelblue'
    )
    axes[1].set_title('Alerts by Check Type')
    axes[1].set_xlabel('Count')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Amount'], bins=50, color='steelblue', alpha=0.7)
axes[0].axvline(df['Amount'].mean(), color='red', linestyle='--', label='Mean')
axes[0].set_title('Claim Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].boxplot(df['Amount'])
axes[1].set_title('Amount Box Plot')
axes[1].set_ylabel('Amount ($)')

plt.tight_layout()
plt.show()

## 9. Export Results

In [ ]:
# Export to Excel
if not fraud_alerts.empty:
    output_file = f'AHFoZ_Fraud_Report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        fraud_alerts.to_excel(writer, sheet_name='All_Alerts', index=False)
        
        severity_summary = fraud_alerts['Severity'].value_counts().reset_index()
        severity_summary.columns = ['Severity', 'Count']
        severity_summary.to_excel(writer, sheet_name='By_Severity', index=False)
        
        check_summary = fraud_alerts['Check'].value_counts().reset_index()
        check_summary.columns = ['Check_Type', 'Count']
        check_summary.to_excel(writer, sheet_name='By_Check_Type', index=False)
    
    print(f"Report exported to: {output_file}")

## 10. Final Summary

In [ ]:
print("FINAL ANALYSIS SUMMARY")
print("="*80)
print(f"Node: {NODE_NAME} ({NODE_ID})")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nDATASET OVERVIEW:")
print(f"  Total Records: {summary_stats['total_records']:,}")
print(f"  Unique Claims: {summary_stats['total_claims']:,}")
print(f"  Unique Providers: {summary_stats['total_providers']:,}")
print(f"  Unique Members: {summary_stats['total_members']:,}")
print(f"  Total Amount: ${summary_stats['total_amount']:,.2f}")
print("\nFRAUD DETECTION RESULTS:")
if not fraud_alerts.empty:
    print(f"  Total Alerts: {len(fraud_alerts):,}")
    print(f"  Total Flagged Amount: ${fraud_alerts['Amount'].sum():,.2f}")
    print(f"  Percentage Flagged: {len(fraud_alerts)/len(df)*100:.2f}%")
else:
    print("  No fraud alerts detected")
print("="*80)